In [10]:
%pip install pandas openpyxl python-dotenv "psycopg[binary]"

In [11]:
from dotenv import load_dotenv
from datetime import datetime
import os
import pandas as pd
import psycopg
from psycopg import sql

load_dotenv()

True

In [12]:
CSV_PATH = "loan_validado.csv"
TABLE_NAME = "public.loan_demo"

# Modo destructivo:
# True  -> elimina la tabla destino, la vuelve a crear según estructura del CSV e inserta.
# False -> conserva la tabla actual y solo controla si se limpian o agregan registros.
DROP_AND_RECREATE_TABLE = True

# Solo se usa cuando DROP_AND_RECREATE_TABLE = False.
# True  -> borra los registros actuales con TRUNCATE antes de insertar.
# False -> agrega los datos al final de la tabla existente.
CLEAR_BEFORE_INSERT = False

def get_connection_params():
    """
    Obtiene los parámetros de conexión a PostgreSQL desde variables de entorno.
    """
    return {
        "host": os.getenv("SUPABASE_DB_HOST"),
        "port": os.getenv("SUPABASE_DB_PORT", "5432"),
        "dbname": os.getenv("SUPABASE_DB_NAME", "postgres"),
        "user": os.getenv("SUPABASE_DB_USER"),
        "password": os.getenv("SUPABASE_DB_PASSWORD"),
        "sslmode": "require",
    }

def validate_env():
    """
    Valida que existan las variables de entorno requeridas para la conexión.
    """
    params = get_connection_params()
    missing = [key for key, value in params.items() if key != "sslmode" and not value]
    if missing:
        raise ValueError("Faltan variables de entorno: " + ", ".join(missing))
    return params

params = validate_env()

safe_params = {
    key: ("***" if key == "password" else value)
    for key, value in params.items()
}

safe_params



def load_dataframe():
    """
    Lee directamente el archivo CSV validado.

    No se realizan transformaciones como:
    - Cambiar nombres de columnas.
    - Convertir columnas a mayúsculas.
    - Forzar tipos numéricos.
    - Reordenar columnas.

    El DataFrame queda con la misma estructura del archivo validado.
    """
    return pd.read_csv(CSV_PATH)

In [13]:
df = load_dataframe()

print(f"Filas leídas desde CSV: {len(df)}")
print(f"Columnas detectadas: {len(df.columns)}")
display(df.head())

list(df.columns)

Filas leídas desde CSV: 44990
Columnas detectadas: 31


,person_gender_female,person_gender_male,person_education_associate,person_education_bachelor,person_education_doctorate,person_education_high_school,person_education_master,person_home_ownership_mortgage,person_home_ownership_other,person_home_ownership_own,...,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status,ratio_monto_ingreso,categoria_credito,fecha_procesamiento
0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,35000.0,16.02,0.49,3.0,561.0,1,0.486462,Muy malo,2026-05-24 07:07:41
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,1000.0,11.14,0.08,2.0,504.0,0,0.081420,Muy malo,2026-05-24 07:07:41
2,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,3.0,5500.0,12.87,0.44,3.0,635.0,1,0.442193,Regular,2026-05-24 07:07:41
3,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,35000.0,15.23,0.44,2.0,675.0,1,0.438855,Bueno,2026-05-24 07:07:41
4,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,35000.0,14.27,0.53,4.0,586.0,1,0.529221,Regular,2026-05-24 07:07:41


['person_gender_female',
 'person_gender_male',
 'person_education_associate',
 'person_education_bachelor',
 'person_education_doctorate',
 'person_education_high_school',
 'person_education_master',
 'person_home_ownership_mortgage',
 'person_home_ownership_other',
 'person_home_ownership_own',
 'person_home_ownership_rent',
 'loan_intent_debtconsolidation',
 'loan_intent_education',
 'loan_intent_homeimprovement',
 'loan_intent_medical',
 'loan_intent_personal',
 'loan_intent_venture',
 'previous_loan_defaults_on_file_no',
 'previous_loan_defaults_on_file_yes',
 'person_age',
 'person_income',
 'person_emp_exp',
 'loan_amnt',
 'loan_int_rate',
 'loan_percent_income',
 'cb_person_cred_hist_length',
 'credit_score',
 'loan_status',
 'ratio_monto_ingreso',
 'categoria_credito',
 'fecha_procesamiento']

In [14]:
with psycopg.connect(**params) as conn:
    with conn.cursor() as cur:
        cur.execute("select version();")
        db_version = cur.fetchone()[0]

print(db_version)

PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit


In [15]:
def table_identifier(table_name):
    """
    Convierte un nombre de tabla tipo 'schema.tabla' en un identificador SQL seguro.
    """
    return sql.Identifier(*table_name.split("."))


def column_identifiers(columns):
    """
    Convierte la lista de columnas del DataFrame en identificadores SQL seguros.
    """
    return sql.SQL(", ").join(sql.Identifier(str(column)) for column in columns)

In [16]:
def infer_postgres_type(series):
    """
    Infiere un tipo PostgreSQL básico a partir del tipo detectado por pandas.
    """
    dtype = series.dtype
    if pd.api.types.is_integer_dtype(dtype):
        return "BIGINT"
    if pd.api.types.is_float_dtype(dtype):
        return "DOUBLE PRECISION"
    if pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    if pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    return "TEXT"


def build_schema_report(df):
    """
    Construye un reporte informativo con la estructura que se usará para crear la tabla.
    """
    return pd.DataFrame({
        "columna_csv": [str(column) for column in df.columns],
        "tipo_pandas_detectado": [str(df[column].dtype) for column in df.columns],
        "tipo_postgresql_sugerido": [infer_postgres_type(df[column]) for column in df.columns],
    })


schema_report_df = build_schema_report(df)
display(schema_report_df)

,columna_csv,tipo_pandas_detectado,tipo_postgresql_sugerido
0,person_gender_female,float64,DOUBLE PRECISION
1,person_gender_male,float64,DOUBLE PRECISION
2,person_education_associate,float64,DOUBLE PRECISION
3,person_education_bachelor,float64,DOUBLE PRECISION
4,person_education_doctorate,float64,DOUBLE PRECISION
5,person_education_high_school,float64,DOUBLE PRECISION
6,person_education_master,float64,DOUBLE PRECISION
7,person_home_ownership_mortgage,float64,DOUBLE PRECISION
8,person_home_ownership_other,float64,DOUBLE PRECISION
9,person_home_ownership_own,float64,DOUBLE PRECISION


In [17]:
def clear_table(conn):
    """
    Elimina todos los registros de la tabla destino y reinicia el autoincremental.
    No elimina la estructura de la tabla.
    """
    query = sql.SQL("truncate table {} restart identity;").format(
        table_identifier(TABLE_NAME)
    )
    with conn.cursor() as cur:
        cur.execute(query)


def drop_table(conn):
    """
    Elimina la tabla destino si existe.
    Advertencia: esta acción elimina la estructura y los registros de la tabla.
    """
    query = sql.SQL("drop table if exists {};").format(
        table_identifier(TABLE_NAME)
    )
    with conn.cursor() as cur:
        cur.execute(query)


def create_table_from_dataframe(conn, df):
    """
    Crea la tabla destino usando las columnas y tipos detectados desde el DataFrame.
    No modifica los datos del DataFrame.
    """
    if df.empty:
        raise ValueError("No se puede crear la tabla porque el DataFrame está vacío.")

    column_definitions = []
    for column in df.columns:
        pg_type = infer_postgres_type(df[column])
        column_definitions.append(
            sql.SQL("{} {}").format(
                sql.Identifier(str(column)),
                sql.SQL(pg_type)
            )
        )

    query = sql.SQL("create table {} ({});").format(
        table_identifier(TABLE_NAME),
        sql.SQL(", ").join(column_definitions)
    )
    with conn.cursor() as cur:
        cur.execute(query)

In [18]:
def insert_dataframe(conn, df):
    """
    Inserta el DataFrame completo en la tabla destino sin modificar sus datos.

    Puntos importantes:
    - Usa las columnas tal como vienen desde el CSV validado.
    - No convierte manualmente tipos de datos.
    - No cambia textos, números ni fechas.
    - Genera el INSERT de forma dinámica usando la estructura del DataFrame.
    """
    columns = list(df.columns)
    placeholders = sql.SQL(", ").join(sql.Placeholder() for _ in columns)

    query = sql.SQL("insert into {} ({}) values ({})").format(
        table_identifier(TABLE_NAME),
        column_identifiers(columns),
        placeholders,
    )

    rows = list(df.itertuples(index=False, name=None))

    with conn.cursor() as cur:
        cur.executemany(query, rows)

    return len(rows)

def add_report_event(report, process_start, step, status="OK", detail="", step_start=None):
    """
    Agrega un evento al reporte de ejecución y lo imprime en pantalla.
    """
    now = datetime.now()
    elapsed_seconds = round((now - process_start).total_seconds(), 2)
    duration_seconds = None
    if step_start is not None:
        duration_seconds = round((now - step_start).total_seconds(), 2)

    report.append({
        "momento": now.strftime("%Y-%m-%d %H:%M:%S"),
        "segundos_desde_inicio": elapsed_seconds,
        "duracion_paso_segundos": duration_seconds,
        "paso": step,
        "estado": status,
        "detalle": detail,
    })
    print(f"[{status}] {step}: {detail}")

def run_load_process():
    """
    Ejecuta el proceso completo de carga hacia Supabase.

    Flujo:
    1. Inicia el reporte de ejecución.
    2. Valida variables de entorno.
    3. Lee el CSV validado.
    4. Conecta a Supabase.
    5. Según parámetros: elimina/recrea tabla, limpia tabla o agrega registros.
    6. Inserta los datos.
    7. Confirma la transacción con commit.
    8. Muestra el reporte final.
    """
    report = []
    process_start = datetime.now()
    inserted_rows = 0

    try:
        add_report_event(report, process_start,
                         step="Inicio del proceso",
                         detail="Se inicia la carga directa hacia Supabase/PostgreSQL.")

        step_start = datetime.now()
        params = validate_env()
        add_report_event(report, process_start,
                         step="Validación de variables de entorno",
                         detail="Credenciales disponibles en archivo .env.",
                         step_start=step_start)

        step_start = datetime.now()
        df = load_dataframe()
        add_report_event(report, process_start,
                         step="Lectura del archivo CSV",
                         detail=f"Archivo leído correctamente: {len(df)} filas y {len(df.columns)} columnas.",
                         step_start=step_start)

        step_start = datetime.now()
        schema_report = build_schema_report(df)
        add_report_event(report, process_start,
                         step="Lectura de estructura del CSV",
                         detail=f"Estructura detectada para {len(schema_report)} columnas.",
                         step_start=step_start)

        step_start = datetime.now()
        with psycopg.connect(**params) as conn:
            add_report_event(report, process_start,
                             step="Conexión a Supabase/PostgreSQL",
                             detail="Conexión establecida correctamente.",
                             step_start=step_start)

            if DROP_AND_RECREATE_TABLE:
                step_start = datetime.now()
                drop_table(conn)
                create_table_from_dataframe(conn, df)
                add_report_event(report, process_start,
                                 step="Recreación de tabla destino",
                                 detail=f"Tabla {TABLE_NAME} eliminada y creada según estructura del CSV.",
                                 step_start=step_start)
            elif CLEAR_BEFORE_INSERT:
                step_start = datetime.now()
                clear_table(conn)
                add_report_event(report, process_start,
                                 step="Limpieza de tabla destino",
                                 detail=f"Tabla {TABLE_NAME} limpiada con TRUNCATE antes de insertar.",
                                 step_start=step_start)
            else:
                add_report_event(report, process_start,
                                 step="Modo de carga incremental",
                                 detail="No se limpia ni recrea la tabla; registros se agregarán al final.")

            step_start = datetime.now()
            inserted_rows = insert_dataframe(conn, df)
            add_report_event(report, process_start,
                             step="Inserción de registros",
                             detail=f"Se insertaron {inserted_rows} filas en {TABLE_NAME}.",
                             step_start=step_start)

            step_start = datetime.now()
            conn.commit()
            add_report_event(report, process_start,
                             step="Confirmación de transacción",
                             detail="Commit ejecutado correctamente.",
                             step_start=step_start)

        add_report_event(report, process_start,
                         step="Fin del proceso",
                         detail="Carga completada exitosamente.")

    except Exception as error:
        add_report_event(report, process_start,
                         step="Error del proceso",
                         status="ERROR",
                         detail=str(error))
        display(pd.DataFrame(report))
        raise

    return inserted_rows, pd.DataFrame(report)


inserted_rows, execution_report_df = run_load_process()
display(execution_report_df)



def read_sample(conn, limit=10):
    """
    Lee una muestra de registros desde la tabla destino para verificar
    que los datos fueron insertados correctamente.
    """
    query = sql.SQL("select * from {} limit {};").format(
        table_identifier(TABLE_NAME),
        sql.Literal(limit),
    )
    with conn.cursor() as cur:
        cur.execute(query)
        rows = cur.fetchall()
        columns = [desc.name for desc in cur.description]
    return pd.DataFrame(rows, columns=columns)


with psycopg.connect(**params) as conn:
    sample_df = read_sample(conn, limit=10)

display(sample_df)

[OK] Inicio del proceso: Se inicia la carga directa hacia Supabase/PostgreSQL.
[OK] Validación de variables de entorno: Credenciales disponibles en archivo .env.
[OK] Lectura del archivo CSV: Archivo leído correctamente: 44990 filas y 31 columnas.
[OK] Lectura de estructura del CSV: Estructura detectada para 31 columnas.
[OK] Conexión a Supabase/PostgreSQL: Conexión establecida correctamente.
[OK] Recreación de tabla destino: Tabla public.loan_demo eliminada y creada según estructura del CSV.
[OK] Inserción de registros: Se insertaron 44990 filas en public.loan_demo.
[OK] Confirmación de transacción: Commit ejecutado correctamente.
[OK] Fin del proceso: Carga completada exitosamente.


,momento,segundos_desde_inicio,duracion_paso_segundos,paso,estado,detalle
0,2026-05-24 07:29:00,0.00,NaN,Inicio del proceso,OK,Se inicia la carga directa hacia Supabase/Post...
1,2026-05-24 07:29:00,0.00,0.00,Validación de variables de entorno,OK,Credenciales disponibles en archivo .env.
2,2026-05-24 07:29:01,0.31,0.31,Lectura del archivo CSV,OK,Archivo leído correctamente: 44990 filas y 31 ...
3,2026-05-24 07:29:01,0.32,0.00,Lectura de estructura del CSV,OK,Estructura detectada para 31 columnas.
4,2026-05-24 07:29:01,0.68,0.36,Conexión a Supabase/PostgreSQL,OK,Conexión establecida correctamente.
5,2026-05-24 07:29:01,0.86,0.18,Recreación de tabla destino,OK,Tabla public.loan_demo eliminada y creada segú...
6,2026-05-24 07:29:05,4.80,3.94,Inserción de registros,OK,Se insertaron 44990 filas en public.loan_demo.
7,2026-05-24 07:29:05,4.86,0.06,Confirmación de transacción,OK,Commit ejecutado correctamente.
8,2026-05-24 07:29:05,4.86,NaN,Fin del proceso,OK,Carga completada exitosamente.


,person_gender_female,person_gender_male,person_education_associate,person_education_bachelor,person_education_doctorate,person_education_high_school,person_education_master,person_home_ownership_mortgage,person_home_ownership_other,person_home_ownership_own,...,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status,ratio_monto_ingreso,categoria_credito,fecha_procesamiento
0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,35000.0,16.02,0.49,3.0,561.0,1,0.486462,Muy malo,2026-05-24 07:07:41
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,1000.0,11.14,0.08,2.0,504.0,0,0.081420,Muy malo,2026-05-24 07:07:41
2,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,3.0,5500.0,12.87,0.44,3.0,635.0,1,0.442193,Regular,2026-05-24 07:07:41
3,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,35000.0,15.23,0.44,2.0,675.0,1,0.438855,Bueno,2026-05-24 07:07:41
4,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,35000.0,14.27,0.53,4.0,586.0,1,0.529221,Regular,2026-05-24 07:07:41
5,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,2500.0,7.14,0.19,2.0,532.0,1,0.193035,Muy malo,2026-05-24 07:07:41
6,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,35000.0,12.42,0.37,3.0,701.0,1,0.374448,Bueno,2026-05-24 07:07:41
7,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,5.0,35000.0,11.11,0.37,4.0,585.0,1,0.366300,Regular,2026-05-24 07:07:41
8,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3.0,35000.0,8.90,0.35,2.0,544.0,1,0.347622,Muy malo,2026-05-24 07:07:41
9,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,1600.0,14.74,0.13,3.0,640.0,1,0.125599,Regular,2026-05-24 07:07:41
